# Charge sharpening : benchmark contre Agrawal et al. 2024, Fig. 7

Expérience `experiments/charge_sharpening/charge_sharpening.py` (issues #137 et #142). Elle reproduit la Fig. 7 de U. Agrawal *et al.*, « Observing quantum measurement collapse as a learnability phase transition », PRX **14**, 041012 (2024), à la taille de l'article `L = 6`, c'est-à-dire `N = 7` ici (l'ancilla `R` compte parmi les `N` qubits).

Le protocole, configuré par `conf/charge_sharpening.yaml` :

1. état initial, Éq. (B1) : `(|0>_R |010101> + |1>_R |010100>) / sqrt(2)`, l'ancilla intriquée avec la charge 3 ou 2 du système, puis brouillé par 5 couches de portes U(1) aléatoires tout-à-tout ;
2. `t = L / 2 = 3` couches de brickwork de portes U(1) aléatoires (une couche = une période complète, liens pairs puis impairs), une instance fixe du circuit par seed comme dans l'article ;
3. entre deux couches (2 rondes), chaque qubit du système est mesuré faiblement avec la force `gamma` : `gamma = 0` n'apprend rien, `gamma = 1` est projectif ;
4. on garde l'entropie d'intrication `S_R` de l'ancilla (en nats, comme l'article) dans chacune des trajectoires.

Les observables de la Fig. 7 sont la moyenne de `S_R` sur les trajectoires, sa variance `mu_2` et le rapport de Binder `B = 1 - mu_4 / (3 mu_2**2)`. Quand `gamma` augmente, les mesures apprennent la charge du système et `S_R` tombe de `ln 2` à 0 : la charge « s'affûte ». La variance culmine là où les trajectoires se partagent entre les deux comportements, et les courbes de Binder de tailles différentes s'y croisent.

Ce notebook lance le balayage avec `experiments/charge_sharpening/charge_sharpening.sh`, relit les dossiers `outputs/charge_sharpening/<date_heure>/N<N>_seed<seed>_g<gamma>/`, trace les trois panneaux par-dessus les courbes numérisées de l'article (`experiments/charge_sharpening/reference/`), puis montre la distribution de `S_R` sur les trajectoires, que la variance et le Binder résument.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so experiments/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Paramètres

Chaque valeur peut être changée ici, ou par la variable d'environnement du même nom (c'est ce que fait le test `tests/test_charge_sharpening_notebook.py` pour un petit balayage). `RUN = False` relit le dernier balayage sans en relancer un.

Avec les valeurs par défaut (`L = 10000` trajectoires comme l'article, 3 instances de circuit, 11 valeurs de `gamma`), le balayage prend quelques minutes. Chaque seed est une instance différente du circuit : la bande autour de nos points est leur dispersion.

In [ ]:
L = os.environ.get("L", "10000")  # trajectories per run, as in the paper
N_VALUES = os.environ.get("N_VALUES", "7")  # total qubits: the paper's L = 6 + R
SEEDS = os.environ.get("SEEDS", "0 1 2")  # one circuit instance per seed
GAMMA_VALUES = os.environ.get(
    "GAMMA_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0"
)
FIXED_INSTANCE = os.environ.get("FIXED_INSTANCE", "true")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Lancer le balayage

In [ ]:
from QELM.utils import latest_sweep, run_with_progress

if RUN:
    env = {
        **os.environ,
        "L": L,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "GAMMA_VALUES": GAMMA_VALUES,
        "FIXED_INSTANCE": FIXED_INSTANCE,
        "OUT_ROOT": str(OUT_ROOT),
    }
    n_runs = len(N_VALUES.split()) * len(SEEDS.split()) * len(GAMMA_VALUES.split())
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/charge_sharpening/charge_sharpening.sh"],
        total=n_runs,
        env=env,
        show=("N", "seed", "gamma"),
    )

sweep_dir = latest_sweep(OUT_ROOT / "charge_sharpening")
print(f"sweep: {sweep_dir}")

## Relire les observables

Une ligne par `(N, gamma)`, moyennée sur les seeds (instances du circuit) : la moyenne ± l'erreur standard entre instances.

In [ ]:
import importlib.util

from QELM.utils import load_sweep, summarize

# The plotting script of the experiment is not a package: import it by path.
spec = importlib.util.spec_from_file_location(
    "plot_charge_sharpening",
    Path("experiments/charge_sharpening/plot_charge_sharpening.py"),
)
plot_cs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_cs)

METRICS = [metric for metric, _, _ in plot_cs.PANELS]  # S_R_mean, S_R_var, binder
rows = load_sweep(sweep_dir, keys=plot_cs.KEYS)
table = summarize(rows, by=("N", "gamma"), metrics=METRICS)
print(f"{len(rows)} runs")
print(f"{'N':>3} {'gamma':>6} {'runs':>4} " + " ".join(f"{m:>20}" for m in METRICS))
for r in table:
    values = " ".join(
        f"{r[f'{m}_mean']:>9.4f} ± {r[f'{m}_sem']:<8.4f}" for m in METRICS
    )
    print(f"{r['N']:>3} {r['gamma']:>6.2f} {r['n_runs']:>4} {values}")

## Fig. 7 : nos points sur ceux de l'article

Trois panneaux, comme la Fig. 7 : `S_R` (nats), sa variance (nats²) et le rapport de Binder, en fonction de `gamma`. Nos points sont les moyennes sur les instances, avec l'erreur standard en barres et la dispersion entre instances en bande ; avec une seule seed, la barre est l'erreur bootstrap sur les trajectoires. Les courbes en tirets sont les données numérisées de l'article, `L = 6` (à comparer) et `L = 20` (pour voir l'effet de taille).

Le Binder n'est tracé que là où la variance dépasse `1e-3` : ailleurs `S_R` est collé à `ln 2` ou à 0, et quelques trajectoires rares rendent `B` très négatif. Sous le graphe, l'écart à la courbe `L = 6` de l'article, interpolé à ses points.

In [ ]:
import json

import matplotlib.pyplot as plt

fig, comparison = plot_cs.draw_sweep(sweep_dir)
figure_path = sweep_dir / "charge_sharpening.png"
fig.savefig(figure_path, dpi=150)
(sweep_dir / "comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
for metric, by_size in comparison.items():
    for size, curves in by_size.items():
        for name, stats in curves.items():
            print(
                f"{metric:>9} {size} vs {name}: "
                f"max |diff| = {stats['max_abs_diff']:.3g}, "
                f"rms = {stats['rms_diff']:.3g} ({stats['n_points']} points)"
            )
print(f"figure: {figure_path}")
plt.show()

## Distribution de `S_R` sur les trajectoires

La moyenne, la variance et le Binder résument cette distribution. À faible `gamma`, presque toutes les trajectoires gardent `S_R ≈ ln 2` ; à fort `gamma`, presque toutes l'ont purifiée (`S_R ≈ 0`). Près de la transition, la distribution s'étale entre les deux : c'est là que la variance culmine et que `B` est positif. Une seule instance (la première seed) et la plus grande taille du balayage.

In [ ]:
import numpy as np

N_hist = max(int(r["N"]) for r in rows)
seed_hist = min(int(r["seed"]) for r in rows if int(r["N"]) == N_hist)
runs = sorted(
    (r for r in rows if int(r["N"]) == N_hist and int(r["seed"]) == seed_hist),
    key=lambda r: r["gamma"],
)
# Up to 5 values of gamma, spread over the sweep.
picks = [
    runs[k] for k in np.unique(np.linspace(0, len(runs) - 1, 5).round().astype(int))
]
hist_fig, hist_axes = plt.subplots(
    1,
    len(picks),
    figsize=(3 * len(picks), 2.8),
    sharey=True,
    squeeze=False,
    constrained_layout=True,
)
bins = np.linspace(0, np.log(2), 31)
for ax, r in zip(hist_axes[0], picks, strict=True):
    S_R = np.load(sweep_dir / r["run"] / "charge_sharpening.npz")["S_R"]
    ax.hist(np.clip(S_R, 0, np.log(2)), bins=bins, color="#2a78d6")
    ax.set(title=f"gamma = {r['gamma']:.2f}", xlabel="S_R (nats)")
    ax.set_yscale("log")
    ax.grid(alpha=0.25, lw=0.5)
hist_axes[0, 0].set_ylabel("trajectories")
hist_fig.suptitle(f"Distribution of S_R, N = {N_hist}, seed {seed_hist}")
plt.show()